In [1]:
import torch
from idinn.sourcing_model import DualSourcingModel
from idinn.dual_controller import DualSourcingNeuralController
from idinn.demand import UniformDemand
import numpy as np
from torch.utils.tensorboard import SummaryWriter

In [2]:
dual_sourcing_model = DualSourcingModel(
    regular_lead_time=2, # lr # Skip # They can order every day, but for a fixed lead time, we can only order every three days, but 0 lead time
    expedited_lead_time=0, # Skip
    regular_order_cost=0, # Non Zero Value
    expedited_order_cost=20, # ce
    holding_cost=5,
    shortage_cost=495, # b
    batch_size=256,
    init_inventory=6,
    demand_generator=UniformDemand(low=0, high=4), # 23.13 ->
)

In [3]:

controller_neural = DualSourcingNeuralController(
    hidden_layers=[128, 64, 32, 16, 8, 4],
    activation=torch.nn.CELU(alpha=1),
    expedited_activation=torch.nn.ReLU(),
    regular_activation=torch.nn.ReLU(), # Change this
)

In [4]:
controller_neural.fit(sourcing_model=dual_sourcing_model, sourcing_periods=100, epochs=3000,
                      validation_sourcing_periods=1000, seed=123)

100%|██████████| 3000/3000 [04:17<00:00, 11.64it/s]


(set(), set())

In [6]:
for name, param in controller_neural.named_parameters():
    print(f"{name} - shape: {param.shape}")
    print(param.data)


hidden_sequence.0.weight - shape: torch.Size([4, 3])
tensor([[-1.2020, -0.0381, -0.2901],
        [ 0.2548, -0.4782,  0.4373],
        [-2.3201, -0.0727, -0.2981],
        [ 0.2959, -0.2007,  0.2291]])
hidden_sequence.0.bias - shape: torch.Size([4])
tensor([ 1.7281, -0.3153,  3.0453, -0.0512])
final_linear.weight - shape: torch.Size([2, 4])
tensor([[-0.4150,  0.3677, -0.1541,  0.1993],
        [ 0.5951, -0.0121,  0.5089,  0.3805]])
final_linear.bias - shape: torch.Size([2])
tensor([-0.1889,  0.2441])


In [5]:
avg_cost = controller_neural.get_average_cost(dual_sourcing_model, sourcing_periods=1000, seed=123)
print(f"Average cost: {avg_cost:.2f}") # 23.76

Average cost: 83.02


In [17]:
controller_neural # 23.07 - +- 0.1/0.2 max variance

DualSourcingNeuralController(
  (activation): CELU(alpha=1)
  (regular_activation): ReLU()
  (expedited_activation): ReLU()
  (hidden_sequence): Sequential(
    (0): Linear(in_features=3, out_features=8, bias=True)
    (1): CELU(alpha=1)
    (2): Linear(in_features=8, out_features=4, bias=True)
    (3): CELU(alpha=1)
  )
  (final_linear): Linear(in_features=4, out_features=2, bias=True)
)

In [ ]:
# 23.13 optimal for 2, 20, 495, (0,4)
# We have 23.76 for 2k epochs - not reproducible - fails, L seed